# Skąd algorytm bierze liczbę do aktualizacji?

**Ćwiczenie porównawcze do modułów 02–06.** Zanim uruchomisz komórkę, zapisz przewidywane wyniki i wskaż dane potrzebne do ich policzenia. Notebook wymaga tylko standardowej biblioteki Pythona.

Przez **cel aktualizacji** rozumiemy liczbę \(y\), w stronę której przesuwamy bieżące oszacowanie \(x\):

\[
x_{\mathrm{nowe}}=x+\alpha(y-x).
\]

W predykcji \(x=V(s)\) oznacza wartość stanu przy zadanej polityce. W sterowaniu \(x=Q(s,a)\) oznacza wartość wykonania akcji w stanie. \(\alpha\) jest krokiem uczenia, a \(\gamma\) dyskontem przyszłych nagród. Pełna aktualizacja Bellmana w programowaniu dynamicznym wstawia wyliczony cel bezpośrednio jako nową wartość.

## Zanim porównasz metody: bandyta, model, tablica ocen

[Otwórz atlas metod tabelarycznych](Atlas_metod_tabelarycznych.html). Wybierz porównanie **Bandyta → sterowanie Monte Carlo** albo **Iteracja polityki ↔ wartości**.

- **Bandyta.** $q_*(a)=\mathbb{E}[R_{t+1}\mid A_t=a]$ to prawdziwa średnia wypłata, a $Q_t(a)$ to jej bieżące oszacowanie. Przy jednoruchowym epizodzie $G_t=r_{t+1}$. Gdy zwiększymy licznik $N(a)$, krok $1/N(a)$ daje dokładną średnią próbkową.
- **Stan i akcja.** Model `P[s][a]` zawiera listę (szansa, następny stan, nagroda, koniec). Wartość `Q[s][a]` to pojedyncza liczba opisująca dalszy zwrot po konkretnym pierwszym ruchu. Przy znanym modelu i zadanej mapie $V$ możesz wyliczyć $q_V(s,a)$, ważąc przez szanse gałęzi składniki $r+\gamma(1-d)V(s')$. Na przykład ocena polityki potem waży akcje przez $\pi(a\mid s)$.
- **Wartość i akcja.** `max` zwraca największą liczbę. `argmax` wskazuje, gdzie ta liczba jest osiągana. Przy remisie jest kilka możliwych akcji.

**Ćwiczenie rachunkowe.** Cztery ramiona mają oceny (4, 4, 2, 0). Dla $\varepsilon=0{,}20$ zapisz prawdopodobieństwa wyboru. Następnie dla robota policz $q_V(H,\text{szukaj})$. Dwie równoprawdopodobne gałęzie dają wypłatę 2 i kierują do $H$ (wysoka energia) lub $L$ (niska energia). Przyjmij $\gamma=0{,}8$, $V(H)=5$ oraz $V(L)=1$. Uruchom komórkę, żeby sprawdzić oba rachunki.


In [ ]:
from math import isclose

ratings = (4, 4, 2, 0)
epsilon = 0.20
max_value = max(ratings)
max_actions = tuple(index for index, value in enumerate(ratings) if value == max_value)
uniform_part = epsilon / len(ratings)
greedy_part = (1 - epsilon) / len(max_actions)
arm_probabilities = tuple(
    uniform_part + (greedy_part if index in max_actions else 0)
    for index in range(len(ratings))
)
print("Szanse ramion 1–4:", tuple(round(p, 2) for p in arm_probabilities))
print("max =", max_value, "| argmax, numeracja od 1:", tuple(index + 1 for index in max_actions))
assert isclose(sum(arm_probabilities), 1)

P_robot = {"H": {"szukaj": [(0.5, "H", 2, False), (0.5, "L", 2, False)]}}
V_robot = {"H": 5, "L": 1}
gamma_robot = 0.8
terms = [
    probability * (reward + gamma_robot * (1 - done) * V_robot[next_state])
    for probability, next_state, reward, done in P_robot["H"]["szukaj"]
]
print("Ważone wkłady obu gałęzi:", [round(term, 2) for term in terms])
print("q_V(H, szukaj) =", round(sum(terms), 2))
assert isclose(sum(terms), 4.4)


## 1. To samo pole, pięć różnych obliczeń

Na śliskim FrozenLake z pola 14 wybieramy „prawo”, a z pola 10 „dół”. Zaczynamy od pierwszej wizyty w 14. Stara tablica podaje \(V(14)=0{,}60\), \(V(10)=0{,}40\), dyskonto \(\gamma=0{,}95\).

**Model po wybraniu „prawo” w 14** dopuszcza trzy skutki po \(1/3\): pole 10 i nagrodę 0, pole 14 i nagrodę 0 oraz pole 15 i nagrodę +1. Pole 15 kończy epizod.

**Zapis jednego epizodu** to \(14\to10\to14\to15\), z nagrodami kolejno 0, 0, +1. To jedna możliwa realizacja tej polityki.

| Metoda | Jak policzyć cel dla \(V(14)\)? |
|---|---|
| Programowanie dynamiczne (DP) | Weź \(r+\gamma V(s')\) z każdej gałęzi pierwszego ruchu i zważ prawdopodobieństwem. |
| TD(0) | Weź jeden zaobserwowany ruch \(14\to10\). Dodaj do jego nagrody \(\gamma V(10)\). |
| 2-krokowe TD | Weź nagrody z dwóch przejść, potem dodaj \(\gamma^2 V(14)\). |
| Monte Carlo pierwszej wizyty | Zdyskontuj trzy rzeczywiste nagrody aż do końca. |
| Zwrot \(\lambda\) z zamrożoną tablicą \(V\) | Dla \(\lambda=0{,}5\) zastosuj wagi \(0{,}5,\ 0{,}25,\ 0{,}25\) do celów z jednego, dwóch i trzech kroków. |

**Przewidź.** Które cele możesz policzyć przed rozpoczęciem epizodu, po pierwszym ruchu, po drugim, a które dopiero po zakończeniu? Ile wynosi zwrot Monte Carlo od drugiej wizyty na 14?

In [ ]:
from math import isclose

gamma = 0.95
V = {14: 0.60, 10: 0.40}
# (prawdopodobieństwo, następny stan, nagroda, koniec)
branches = ((1/3, 10, 0, False), (1/3, 14, 0, False), (1/3, 15, 1, True))
# (stan, następny stan, nagroda, koniec), jeden z możliwych epizodów
episode = ((14, 10, 0, False), (10, 14, 0, False), (14, 15, 1, True))

def one_step_target(reward, next_state, done):
    return reward if done else reward + gamma * V[next_state]

def n_step_target(start, n):
    if not 0 <= start < len(episode) or n < 1:
        raise ValueError("Wybierz istniejące przejście i co najmniej jeden krok.")
    reward_sum = 0.0
    for k, (_, next_state, reward, done) in enumerate(episode[start:start+n]):
        reward_sum += gamma**k * reward
        if done:
            return reward_sum
    return reward_sum + gamma**n * V[next_state]

def show_table(rows):
    for method, value in rows.items():
        print(f"{method:28s} {value:.5f}".replace(".", ","))

dp_target = sum(p * one_step_target(r, sp, done) for p, sp, r, done in branches)
td_target, two_step, mc_target = (n_step_target(0, n) for n in (1, 2, 3))
lam = 0.5
lambda_return = (1-lam)*td_target + (1-lam)*lam*two_step + lam**2*mc_target

predictions = {
    "DP, pełny model": dp_target,
    "TD(0)": td_target,
    "2-krokowe TD": two_step,
    "MC, pierwsza wizyta": mc_target,
    "Zwrot λ": lambda_return,
}
show_table(predictions)
print(f"MC od drugiej wizyty w 14: {n_step_target(2, 1):g}")

In [ ]:
expected = (0.65, 0.38, 0.5415, 0.9025, 0.551)
assert all(isclose(value, answer) for value, answer in zip(predictions.values(), expected))
assert isclose(n_step_target(2, 1), 1.0)
print("Pięć celów i zwrot z drugiej wizyty zgodne z rachunkiem.")

**Co sprawdzamy dalej.** W module **02** (RL_02_DP.ipynb) stosujemy pełny rachunek w każdym stanie. Ocena polityki uśrednia wartości akcji według tej polityki, poprawa wybiera akcje zachłannie względem oceny, a iteracja wartości bierze maksimum przy każdym przebiegu. W module **03** (RL_03_MC_TD0.ipynb) porównujemy predykcje MC i TD(0) na wspólnych epizodach z rozwiązaniem referencyjnym DP. W module **05** (RL_05_Wielokrokowe.ipynb) zmieniamy długość horyzontu i wprowadzamy ślady kwalifikowalności. Zwrot \(\lambda\) policzony powyżej używa jednej zamrożonej tablicy \(V\). Aktualizowanie \(V\) w trakcie epizodu daje inny rachunek.

W module 03 pojawia się też sterowanie Monte Carlo. Tam cały epizod służy aktualizacjom \(Q(s,a)\), a zmiana \(Q\) pozwala poprawić politykę wyboru akcji.

## 2. Ten sam ruch, trzy cele sterowania

Po kroku na planszy z urwiskiem mamy nagrodę \(r=-1\) i \(\gamma=0{,}90\). W następnym stanie wartości czterech akcji w kolejności lewo, dół, prawo, góra to \(4,1,2,0\). Zachowanie jest \(\varepsilon\)-zachłanne przy \(\varepsilon=0{,}20\). Oznacza to wybór najlepszej akcji z prawdopodobieństwem \(1-\varepsilon\) oraz losowanie ze wszystkich czterech z prawdopodobieństwem \(\varepsilon\). W tej realizacji następna wylosowana akcja to „prawo”.

**Przewidź.** Policz trzy cele \(r+\gamma z\). Jakie są cztery prawdopodobieństwa akcji? Które algorytmy zmienią cel, gdy z zachowania wypadnie „góra” zamiast „prawo”? Co zostanie w celu, jeżeli ruch kończy epizod?

- **SARSA** bierze \(z=Q(s',a')\) dla faktycznie wybranej \(a'\).
- **Expected SARSA** uśrednia \(z=Q(s',a)\) według wskazanej polityki docelowej. W tym ćwiczeniu jest nią polityka zachowania.
- **Q-learning** bierze \(z=\max_a Q(s',a)\). Zachowanie może nadal eksplorować.

In [ ]:
reward, discount, eps = -1, 0.90, 0.20
actions = ("lewo", "dół", "prawo", "góra")
q_next = dict(zip(actions, (4, 1, 2, 0)))
actual_action = "prawo"
best_value = max(q_next.values())
best_actions = [a for a in actions if q_next[a] == best_value]
prob = {
    a: eps / len(actions) + ((1-eps)/len(best_actions) if a in best_actions else 0)
    for a in actions
}

def control_targets(done=False):
    if done:
        return dict.fromkeys(("SARSA", "Expected SARSA", "Q-learning"), reward)
    future = {
        "SARSA": q_next[actual_action],
        "Expected SARSA": sum(prob[a] * q_next[a] for a in actions),
        "Q-learning": best_value,
    }
    return {name: reward + discount * value for name, value in future.items()}

print("Szanse wyboru:", {a: round(prob[a], 2) for a in actions})
targets = control_targets()
show_table(targets)
print("Cele po zakończeniu:", control_targets(done=True))

In [ ]:
assert isclose(sum(prob.values()), 1.0)
assert all(isclose(value, answer) for value, answer in zip(targets.values(), (0.8, 2.195, 2.6)))
assert set(control_targets(done=True).values()) == {reward}
print("Cele sterowania i przypadek terminalny zgodne z rachunkiem.")

**Co sprawdzamy dalej.** W module **04** (RL_04_Sterowanie_tabelaryczne.ipynb) przechodzimy od jednego takiego celu do uczenia trasy. Oddzielamy zwrot podczas eksploracji od wyniku zamrożonej polityki zachłannej. Prawdopodobieństwo wyboru akcji wynika z polityki, a prawdopodobieństwo wyniku już wybranej akcji z modelu środowiska.

## 3. Kto wybiera i skąd bierze się przejście?

**Double Q-learning.** Niech pierwsza tablica ocenia dwie akcje w kolejnym stanie na \(6\) i \(4\), a druga na \(3\) i \(5\). Aktualizujemy pierwszą tablicę. Dla \(r=0\) i \(\gamma=0{,}9\) wskaż, która tablica wybiera akcję i która podstawia wartość do celu.

**Dyna-Q.** Agent wykonał 12 rzeczywistych ruchów. Każdy krok dał jedną aktualizację \(Q\) z obserwacji i zapis przejścia. Po każdym ruchu agent zrobił dodatkowo 5 aktualizacji z poznanego modelu. Ile razy zapytał środowisko, a ile razy zaktualizował \(Q\)? Gdy podstawisz to samo przejście i tę samą tablicę \(Q\), rachunek pojedynczej aktualizacji jest taki sam dla próbki rzeczywistej i modelowej.

In [ ]:
A, B = (6, 4), (3, 5)
a_selected = max(range(len(A)), key=lambda a: A[a])
double_target = 0.9 * B[a_selected]
ordinary_target = 0.9 * A[a_selected]

real_steps, planning_per_step = 12, 5
planning_updates = real_steps * planning_per_step
total_updates = real_steps + planning_updates

show_table({"Zwykły cel Q": ordinary_target, "Cel Double Q": double_target})
print(f"Dyna-Q: {real_steps} ruchów, {planning_updates} aktualizacji modelowych, {total_updates} aktualizacje Q.")

assert isclose(double_target, 2.7)
assert (real_steps, planning_updates, total_updates) == (12, 60, 72)

**Co sprawdzamy dalej.** W module **06** (RL_06_Double_Q_Dyna_Q.ipynb) badamy te dwa mechanizmy osobno. W Double Q oceniamy wpływ maksimum przy zaszumionych wartościach. W Dyna-Q porównujemy wynik przy tym samym budżecie rzeczywistych ruchów i różnej liczbie aktualizacji z modelu.

### Zadanie na koniec

Wybierz najbliższą parę metod. W jednym zdaniu opisz **która jedna rzecz zmieniła się w aktualizacji** i **co utrzymujesz stałe, żeby to sprawdzić**. Następnie skopiuj komórkę rachunku na koniec notebooka. Zmień jedną liczbę, zapisz hipotezę i sprawdź ją kodem. Komórki z asercjami sprawdzają wartości dla danych podanych w treści.